# Demo Notebook: GitHub-Loaded Perception And RL Models

This notebook contains only demo and inference code. It downloads the saved ResNet18 checkpoint, class mean embeddings, and the exported DQN policy from GitHub, downloads a small fixed test subset from Kaggle, runs perception inference on that Kaggle subset, visualizes predictions, and then runs a deterministic rollout with the saved RL policy on the warehouse environment.

In [ ]:
from pathlib import Path
import json
import math

import kagglehub
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
import requests
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torchvision.models import resnet18

REPO_OWNER = "avasenin-14"
REPO_NAME = "advanced-deep-learning"
GITHUB_REF = "main"
DATASET_ID = "machowe/warehouse-object-detection-dataset"
ARTIFACT_DIR = Path("artifacts") / "demo"
DATA_DIR = Path("data") / "warehouse_demo_subset"
ARTIFACT_SPECS = {
    "checkpoint": "raw_multilabel_resnet18_best.pth",
    "embeddings": "class_mean_embeddings_resnet18.pt",
    "rl_policy": "warehouse_dqn_best_avg_reward.pth",
}
ARTIFACT_MIN_BYTES = {
    "checkpoint": 40_000_000,
    "embeddings": 5_000,
    "rl_policy": 10_000,
}
DEMO_SAMPLE_STEMS = [
    "warehouse_000001",
    "warehouse_000003",
    "warehouse_000004",
    "warehouse_000006",
    "warehouse_000013",
    "warehouse_000015",
]
BACKGROUND_NAMES = {"wall", "ceiling", "floor", "pillar"}
NAME_ALIASES = {"barel": "barrel"}
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
IMAGE_SIZE = 224
BATCH_SIZE = 16
WORKSPACE_ROOT = Path.cwd()


def github_raw_url(path: str) -> str:
    return f"https://github.com/{REPO_OWNER}/{REPO_NAME}/raw/{GITHUB_REF}/{path}"



def local_artifact_fallback(relative_path: str):
    candidate = WORKSPACE_ROOT / relative_path
    if candidate.exists() and candidate.is_file():
        return candidate
    return None



def local_dataset_fallback(relative_path: str):
    candidate = WORKSPACE_ROOT / "data" / "warehouse" / relative_path
    if candidate.exists() and candidate.is_file():
        return candidate
    return None



def download_file(
    url: str,
    destination: Path,
    fallback_path: Path | None = None,
    minimum_size_bytes: int = 1,
) -> Path:
    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.exists() and destination.stat().st_size >= minimum_size_bytes:
        print(f"Reusing cached artifact: {destination}")
        return destination
    if destination.exists() and destination.stat().st_size < minimum_size_bytes:
        destination.unlink()

    temp_path = destination.with_suffix(destination.suffix + ".part")
    if temp_path.exists():
        temp_path.unlink()

    print(f"Downloading {destination.name} from {url}")
    try:
        with requests.get(url, stream=True, timeout=60, allow_redirects=True) as response:
            response.raise_for_status()
            expected_size = int(response.headers.get("content-length") or 0)
            bytes_written = 0
            with open(temp_path, "wb") as file:
                for chunk in response.iter_content(chunk_size=1024 * 1024):
                    if chunk:
                        file.write(chunk)
                        bytes_written += len(chunk)
        if expected_size and bytes_written != expected_size:
            raise IOError(
                f"Incomplete download for {destination.name}: expected {expected_size} bytes, got {bytes_written}."
            )
        if temp_path.stat().st_size < minimum_size_bytes:
            raise IOError(
                f"Downloaded file is smaller than expected for {destination.name}: {temp_path.stat().st_size} bytes."
            )
        temp_path.replace(destination)
        return destination
    except Exception as exc:
        if temp_path.exists():
            temp_path.unlink()
        if destination.exists() and destination.stat().st_size >= minimum_size_bytes:
            print(f"Download completed despite client error, using: {destination}")
            return destination
        if fallback_path is not None and fallback_path.exists():
            print(f"Download failed, using local fallback: {fallback_path}")
            return fallback_path
        raise RuntimeError(f"Could not download {url}") from exc



def download_dataset_file(relative_path: str, output_dir: Path) -> Path:
    destination = output_dir / relative_path
    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.exists() and destination.stat().st_size > 0:
        print(f"Reusing cached dataset file: {destination}")
        return destination

    fallback_path = local_dataset_fallback(relative_path)
    print(f"Downloading {relative_path} from Kaggle")
    try:
        result = Path(
            kagglehub.dataset_download(
                DATASET_ID,
                path=relative_path,
                output_dir=str(output_dir),
            )
        )
        if result.exists() and result.stat().st_size > 0:
            return result
        if destination.exists() and destination.stat().st_size > 0:
            return destination
        raise FileNotFoundError(f"Kaggle path downloaded but no file was found: {relative_path}")
    except Exception as exc:
        if destination.exists() and destination.stat().st_size > 0:
            print(f"Using downloaded dataset file: {destination}")
            return destination
        if fallback_path is not None and fallback_path.exists():
            print(f"Kaggle download failed, using local fallback: {fallback_path}")
            return fallback_path
        raise RuntimeError(f"Could not download Kaggle path: {relative_path}") from exc


print(f"Using device: {DEVICE}")

In [ ]:
artifact_paths = {}
for name, relative_path in ARTIFACT_SPECS.items():
    artifact_paths[name] = download_file(
        github_raw_url(relative_path),
        ARTIFACT_DIR / relative_path,
        fallback_path=local_artifact_fallback(relative_path),
        minimum_size_bytes=ARTIFACT_MIN_BYTES[name],
    )

dataset_relative_paths = ["warehouse_detection_dataset/class_mapping.json"]
for stem in DEMO_SAMPLE_STEMS:
    dataset_relative_paths.append(
        f"warehouse_detection_dataset/raw/test/images/{stem}.png"
    )
    dataset_relative_paths.append(
        f"warehouse_detection_dataset/raw/test/annotations/{stem}_labels.json"
    )

dataset_files = {
    relative_path: download_dataset_file(relative_path, DATA_DIR)
    for relative_path in dataset_relative_paths
}
class_mapping_path = dataset_files["warehouse_detection_dataset/class_mapping.json"]
dataset_root = Path(class_mapping_path).parent
with open(class_mapping_path, "r", encoding="utf-8") as file:
    class_mapping = json.load(file)

demo_samples = [
    {
        "stem": stem,
        "image_path": dataset_files[
            f"warehouse_detection_dataset/raw/test/images/{stem}.png"
        ],
        "labels_path": dataset_files[
            f"warehouse_detection_dataset/raw/test/annotations/{stem}_labels.json"
        ],
    }
    for stem in DEMO_SAMPLE_STEMS
]

print("Artifact paths:")
for name, path in artifact_paths.items():
    print(f"  {name}: {path} ({path.stat().st_size / (1024 ** 2):.2f} MB)")
print(f"Dataset root: {dataset_root}")
print(f"Demo sample count: {len(demo_samples)}")
print("Demo sample stems:", DEMO_SAMPLE_STEMS)
print(f"Classes listed in mapping: {len(class_mapping['classes'])}")

In [ ]:
def load_torch_payload(primary_path: Path, fallback_path: Path | None = None):
    try:
        return torch.load(primary_path, map_location="cpu", weights_only=False), Path(primary_path)
    except Exception as exc:
        if fallback_path is not None and Path(fallback_path).exists() and Path(fallback_path) != Path(primary_path):
            print(f"Failed to load {primary_path}, using fallback: {fallback_path}")
            return torch.load(fallback_path, map_location="cpu", weights_only=False), Path(fallback_path)
        raise RuntimeError(f"Could not load PyTorch payload from {primary_path}") from exc


checkpoint, checkpoint_source = load_torch_payload(
    Path(artifact_paths["checkpoint"]),
    fallback_path=local_artifact_fallback(ARTIFACT_SPECS["checkpoint"]),
)
embedding_payload, embedding_source = load_torch_payload(
    Path(artifact_paths["embeddings"]),
    fallback_path=local_artifact_fallback(ARTIFACT_SPECS["embeddings"]),
)
rl_payload, rl_policy_source = load_torch_payload(
    Path(artifact_paths["rl_policy"]),
    fallback_path=local_artifact_fallback(ARTIFACT_SPECS["rl_policy"]),
)
class_names = [str(name) for name in checkpoint["class_names"]]
name_to_id = {name: idx for idx, name in enumerate(class_names)}
threshold = float(checkpoint.get("threshold", 0.5))



def parse_raw_class_list(labels_json_path: Path):
    with open(labels_json_path, "r", encoding="utf-8") as file:
        raw_labels = json.load(file)

    labels = set()
    for item in raw_labels.values():
        raw_value = item.get("class", "")
        tokens = [token.strip() for token in str(raw_value).split(",") if token.strip()]
        for token in tokens:
            if token == "data":
                continue
            token = NAME_ALIASES.get(token, token)
            if token in BACKGROUND_NAMES:
                continue
            if token in name_to_id:
                labels.add(name_to_id[token])
    return sorted(labels)


class DemoSubsetDataset(Dataset):
    def __init__(self, sample_records, image_size: int = 224):
        self.sample_records = list(sample_records)
        self.transform = transforms.Compose([
            transforms.Resize((image_size, image_size)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ])

    def __len__(self):
        return len(self.sample_records)

    def __getitem__(self, idx):
        record = self.sample_records[idx]
        image = Image.open(record["image_path"]).convert("RGB")
        image_tensor = self.transform(image)
        target = torch.zeros(len(class_names), dtype=torch.float32)
        for class_id in parse_raw_class_list(record["labels_path"]):
            target[class_id] = 1.0
        return image_tensor, target, str(record["image_path"])


demo_ds = DemoSubsetDataset(sample_records=demo_samples, image_size=IMAGE_SIZE)
demo_loader = DataLoader(
    demo_ds,
    batch_size=min(BATCH_SIZE, len(demo_ds)),
    shuffle=False,
)

print(f"Checkpoint strategy: {checkpoint.get('strategy')}")
print(f"Checkpoint validation F1: {float(checkpoint.get('best_val_f1', float('nan'))):.4f}")
print(f"Threshold: {threshold}")
print(f"Checkpoint source: {checkpoint_source}")
print(f"Embedding source:  {embedding_source}")
print(f"RL policy source:  {rl_policy_source}")
print(
    f"RL policy variant: {rl_payload.get('variant')} | avg_reward={float(rl_payload.get('eval', {}).get('avg_reward', float('nan'))):.4f} | "
    f"success_rate={float(rl_payload.get('eval', {}).get('success_rate', float('nan'))):.4f}"
)
print(f"Demo samples: {len(demo_ds)}")
print(f"Embedding classes: {list(embedding_payload['classes'])}")

In [ ]:
model = resnet18(weights=None)
model.fc = nn.Linear(model.fc.in_features, len(class_names))
model.load_state_dict(checkpoint["model_state_dict"])
model = model.to(DEVICE)
model.eval()


def decode_labels(binary_row: torch.Tensor):
    return [class_names[idx] for idx in torch.where(binary_row > 0.5)[0].tolist()]


preview_examples = []
tp = 0.0
fp = 0.0
fn = 0.0

with torch.no_grad():
    for images, targets, image_paths in demo_loader:
        logits = model(images.to(DEVICE))
        probabilities = torch.sigmoid(logits).cpu()
        predictions = (probabilities >= threshold).float()

        tp += ((predictions == 1) & (targets == 1)).sum().item()
        fp += ((predictions == 1) & (targets == 0)).sum().item()
        fn += ((predictions == 0) & (targets == 1)).sum().item()

        for row_index, image_path in enumerate(image_paths):
            top_indices = torch.topk(probabilities[row_index], k=min(5, len(class_names))).indices.tolist()
            preview_examples.append(
                {
                    "path": image_path,
                    "true_labels": decode_labels(targets[row_index]),
                    "pred_labels": decode_labels(predictions[row_index]),
                    "top_scores": [
                        (class_names[idx], float(probabilities[row_index, idx]))
                        for idx in top_indices
                    ],
                }
            )

precision = tp / (tp + fp + 1e-8)
recall = tp / (tp + fn + 1e-8)
f1 = 2 * precision * recall / (precision + recall + 1e-8)

print(f"Demo subset micro precision: {precision:.4f}")
print(f"Demo subset micro recall:    {recall:.4f}")
print(f"Demo subset micro F1:        {f1:.4f}")
print(f"Preview images:              {len(preview_examples)}")

In [ ]:
columns = 2
rows = math.ceil(len(preview_examples) / columns)
fig, axes = plt.subplots(rows, columns, figsize=(14, 5 * rows))
axes = np.atleast_1d(axes).ravel()

for ax, example in zip(axes, preview_examples):
    image = Image.open(example["path"]).convert("RGB")
    ax.imshow(image)
    true_text = ", ".join(example["true_labels"]) or "none"
    pred_text = ", ".join(example["pred_labels"]) or "none"
    top_text = ", ".join(
        f"{label}={score:.2f}" for label, score in example["top_scores"][:3]
    )
    ax.set_title(
        f"{Path(example['path']).name}\ntrue: {true_text}\npred: {pred_text}\ntop: {top_text}",
        fontsize=9,
    )
    ax.axis("off")

for ax in axes[len(preview_examples):]:
    ax.axis("off")

plt.tight_layout()
plt.show()

## Reinforcement-Learning Demo

The next cells load the DQN checkpoint exported from the reinforcement notebook. Because the current evaluation metrics are non-discriminative for the saved policies (`success_rate = 0.0` and `avg_length = max_steps`), the checkpoint is selected by mean training reward over the last 50 episodes. The evaluation issue still needs separate investigation.

In [ ]:
from WarehouseEnv import DEFAULT_REWARD_CONFIG, WarehouseEnv


class QNetwork(nn.Module):
    def __init__(self, input_dim: int, output_dim: int, hidden_dim: int = 128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, output_dim),
        )

    def forward(self, x):
        return self.net(x)


ACTION_LABELS = {
    0: "up",
    1: "down",
    2: "left",
    3: "right",
}

policy_expected_class_order = [
    str(name) for name in rl_payload.get("expected_class_order", ["Floor", "Wall", "Pallet", "Sign"])
]
embedding_classes = [str(name) for name in embedding_payload["classes"]]
class_to_index = {name.lower(): idx for idx, name in enumerate(embedding_classes)}
ordered_rows = [class_to_index[name.lower()] for name in policy_expected_class_order]
ordered_embeddings = embedding_payload["mean_embeddings"][ordered_rows]
if torch.is_tensor(ordered_embeddings):
    ordered_embeddings = ordered_embeddings.detach().cpu().numpy()

embeddings_dict = {
    idx: np.asarray(ordered_embeddings[idx], dtype=np.float32)
    for idx in range(len(policy_expected_class_order))
}

grid_map = np.asarray(rl_payload["grid_map"], dtype=np.int64)
reward_config = dict(rl_payload.get("reward_config", DEFAULT_REWARD_CONFIG))
max_steps = int(rl_payload.get("max_steps", 60))

env = WarehouseEnv(
    grid_map=grid_map,
    embeddings_dict=embeddings_dict,
    reward_config=reward_config,
    max_steps=max_steps,
    start_pos=(0, 0),
    random_start=False,
)

policy = QNetwork(
    input_dim=int(rl_payload["state_dim"]),
    output_dim=int(rl_payload["action_dim"]),
    hidden_dim=int(rl_payload["hidden_dim"]),
)
policy.load_state_dict(rl_payload["state_dict"])
policy = policy.to(DEVICE)
policy.eval()

observation, info = env.reset(seed=int(rl_payload.get("seed", 42)))
trajectory = [
    {
        "step": 0,
        "pos": tuple(env.agent_pos.tolist()),
        "action": "start",
        "reward": 0.0,
        "distance": int(info["distance_to_target"]),
        "hit_wall": False,
    }
]
total_reward = 0.0
terminated = False
truncated = False

with torch.no_grad():
    for step in range(max_steps):
        state_tensor = torch.as_tensor(observation, dtype=torch.float32, device=DEVICE).unsqueeze(0)
        q_values = policy(state_tensor)
        action = int(torch.argmax(q_values, dim=1).item())
        next_observation, reward, terminated, truncated, step_info = env.step(action)
        total_reward += float(reward)
        trajectory.append(
            {
                "step": step + 1,
                "pos": tuple(env.agent_pos.tolist()),
                "action": ACTION_LABELS[action],
                "reward": float(reward),
                "distance": int(step_info["distance_to_target"]),
                "hit_wall": bool(step_info["hit_wall"]),
            }
        )
        observation = next_observation
        if terminated or truncated:
            break

positions = np.asarray([entry["pos"] for entry in trajectory], dtype=np.int64)
target_y, target_x = np.argwhere(grid_map == 2)[0]

print(rl_payload.get("selection_note", ""))
print(
    f"Loaded RL policy: {rl_payload['variant']} | avg_reward={float(rl_payload['eval']['avg_reward']):.4f} | "
    f"avg_length={float(rl_payload['eval']['avg_length']):.2f} | success_rate={float(rl_payload['eval']['success_rate']):.4f}"
)
print(f"Policy source: {rl_policy_source}")
print(f"Observation shape: {observation.shape}")
print(f"Rollout steps executed: {len(trajectory) - 1}")
print(f"Rollout total reward: {total_reward:.4f}")
print(f"Terminated: {terminated} | Truncated: {truncated}")
print("Step-by-step trajectory:")
for entry in trajectory:
    print(
        f"step={entry['step']:>2} | pos={entry['pos']} | action={entry['action']:<5} | "
        f"reward={entry['reward']:+.4f} | distance={entry['distance']} | hit_wall={entry['hit_wall']}"
    )

plt.figure(figsize=(5, 5))
plt.imshow(grid_map, cmap="viridis")
plt.plot(positions[:, 0], positions[:, 1], color="white", linewidth=2, marker="o", markersize=5, label="policy path")
plt.scatter([positions[0, 0]], [positions[0, 1]], c="cyan", marker="o", s=120, label="start")
plt.scatter([positions[-1, 0]], [positions[-1, 1]], c="orange", marker="X", s=140, label="end")
plt.scatter([target_x], [target_y], c="red", marker="*", s=240, label="pallet")
plt.title("Deterministic rollout of saved DQN policy")
plt.xticks(range(grid_map.shape[1]))
plt.yticks(range(grid_map.shape[0]))
plt.grid(color="white", alpha=0.35)
plt.legend(loc="upper left")
plt.show()